## AQI classification with new SNN encoding


## Preparing data
The dataset used is hourly air quality data (2015 - 2020) from various measuring stations across India: https://www.kaggle.com/rohanrao/air-quality-data-in-india

We'll use one city (Thiruvananthapuram in Kerala) that has two stations and compare it with the actual AQI values present in the data at station, city, hour and day level to confirm the calculations are correct.


In [11]:
## importing packages
import numpy as np
import pandas as pd

In [12]:
## defining constants
PATH_STATION_HOUR = "./data/station_hour.csv"
PATH_STATIONS = "./data/stations.csv"

In [13]:
## importing data and subsetting the station
df = pd.read_csv(PATH_STATION_HOUR, parse_dates=["Datetime"], low_memory=False)
stations = pd.read_csv(PATH_STATIONS)

df = df.merge(stations, on="StationId")

df.sort_values(["StationId", "Datetime"], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Loaded {len(df):,} rows, {df.StationId.nunique()} stations")

Loaded 2,589,083 rows, 110 stations


# SNN Part

SNN-focused pipeline:

- foundation 24-hour window contract
- breakpoint-aware Gaussian population encoding (absolute channels)
- deterministic latency spikes (no random rate coding)
- population-path SNN training and evaluation


In [14]:
## SNN imports
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

import snntorch as snn


### Foundation data contract
Build a time-safe split before windowing, then define one 24-hour station sample contract with: raw pollutant values [24, 7], missing-mask [24, 7], hour/month metadata per timestep, and label.

This notebook is population-path focused and uses the new deterministic Gaussian absolute encoding pipeline.

In [15]:
FEATURES = ["PM2.5", "PM10", "SO2", "NOx", "NH3", "CO", "O3"]
BUCKET_TO_INT = {
    "Good": 0,
    "Satisfactory": 1,
    "Moderate": 2,
    "Poor": 3,
    "Very Poor": 4,
    "Severe": 5,
}
INT_TO_BUCKET = {v: k for k, v in BUCKET_TO_INT.items()}
WINDOW_SIZE = 24

# DEV mode to run fast iteration
DEV_MODE = True

# Selection toggle for training samples:
# true - all data (like it is right now);
# false - only rows that are incomplete (taking into account that some stations have missing pollutant readings in all
# rows, so if row is incomplete but these pollutants are missing in all other rows of this station, then it's also complete, since the stastion doesn't have sensors for this
# pollutant)
ALL_DATA_SELECTION = False

POPULATION_STEPS_FULL = 8
POPULATION_STEPS_DEV = 4
EPOCHS_POP_FULL = 10
EPOCHS_POP_DEV = 3
TRAIN_BATCH_SIZE_FULL = 256
TRAIN_BATCH_SIZE_DEV = 512
EVAL_BATCH_SIZE_FULL = 512
EVAL_BATCH_SIZE_DEV = 512
MAX_TRAIN_BATCHES_DEV = 80
MAX_EVAL_BATCHES_DEV = 30
NUM_WORKERS = min(4, max(1, os.cpu_count() or 1))

POPULATION_STEPS_ACTIVE = POPULATION_STEPS_DEV if DEV_MODE else POPULATION_STEPS_FULL
EPOCHS_POP_ACTIVE = EPOCHS_POP_DEV if DEV_MODE else EPOCHS_POP_FULL
TRAIN_BATCH_SIZE_ACTIVE = TRAIN_BATCH_SIZE_DEV if DEV_MODE else TRAIN_BATCH_SIZE_FULL
EVAL_BATCH_SIZE_ACTIVE = EVAL_BATCH_SIZE_DEV if DEV_MODE else EVAL_BATCH_SIZE_FULL
MAX_TRAIN_BATCHES_ACTIVE = MAX_TRAIN_BATCHES_DEV if DEV_MODE else None
MAX_EVAL_BATCHES_ACTIVE = MAX_EVAL_BATCHES_DEV if DEV_MODE else None

# Use existing AQI bucket from dataset, or precomputed fallback if present.
if "AQI_bucket_calculated" in df.columns:
    target_col = "AQI_bucket_calculated"
elif "AQI_Bucket" in df.columns:
    target_col = "AQI_Bucket"
else:
    raise ValueError("Dataset must contain AQI_Bucket or AQI_bucket_calculated")

base_df = df[["StationId", "Datetime"] + FEATURES + [target_col]].copy()
base_df = base_df.rename(columns={target_col: "target_bucket"})
base_df["Datetime"] = pd.to_datetime(base_df["Datetime"], errors="coerce")
base_df = base_df.dropna(subset=["StationId", "Datetime", "target_bucket"])
base_df = base_df[base_df["target_bucket"].astype(str) != "0"]

base_df["label"] = base_df["target_bucket"].map(BUCKET_TO_INT)
base_df = base_df.dropna(subset=["label"]).copy()
base_df["label"] = base_df["label"].astype(int)
base_df = base_df.sort_values(["Datetime", "StationId"]).reset_index(drop=True)

print(f"Using target bucket column: {target_col}")
print(f"Rows with valid AQI bucket: {len(base_df):,}")
print(base_df["target_bucket"].value_counts())
print(
    f"Runtime profile -> DEV_MODE={DEV_MODE}, steps={POPULATION_STEPS_ACTIVE}, epochs={EPOCHS_POP_ACTIVE}, train_bs={TRAIN_BATCH_SIZE_ACTIVE}, eval_bs={EVAL_BATCH_SIZE_ACTIVE}, train_batch_cap={MAX_TRAIN_BATCHES_ACTIVE}, workers={NUM_WORKERS}"
)
print(f"Datetime span: {base_df['Datetime'].min()} -> {base_df['Datetime'].max()}")

Using target bucket column: AQI_Bucket
Rows with valid AQI bucket: 2,018,893
target_bucket
Moderate        675008
Satisfactory    530164
Very Poor       301150
Poor            239990
Good            152113
Severe          120468
Name: count, dtype: int64
Runtime profile -> DEV_MODE=True, steps=4, epochs=3, train_bs=512, eval_bs=512, train_batch_cap=80, workers=4
Datetime span: 2015-01-01 16:00:00 -> 2020-07-01 00:00:00


## Foundation: time-safe split before windowing + sample contract

This cell also supports two selection modes for experiments:
- all windows (default)
- only incomplete windows after excluding station-level structural missing pollutants (likely no sensor).

In [16]:
## Split by time BEFORE windowing (train/val/test = 80/10/10 by timestamp)
unique_times = np.sort(base_df["Datetime"].unique())
train_cut_idx = int(len(unique_times) * 0.8)
val_cut_idx = int(len(unique_times) * 0.9)
train_cutoff = unique_times[max(train_cut_idx - 1, 0)]
val_cutoff = unique_times[max(val_cut_idx - 1, 0)]

split_train_df = base_df[base_df["Datetime"] <= train_cutoff].copy()
split_val_df = base_df[
    (base_df["Datetime"] > train_cutoff) & (base_df["Datetime"] <= val_cutoff)
].copy()
split_test_df = base_df[base_df["Datetime"] > val_cutoff].copy()

# Station-level structural missingness (all rows missing for pollutant => likely no sensor).
station_observed_counts = base_df.groupby("StationId")[FEATURES].count()
station_structural_missing = station_observed_counts.eq(0)
station_structural_missing_map = {
    station_id: row.to_numpy(dtype=bool)
    for station_id, row in station_structural_missing.iterrows()
}

class AQIWindowDataset(torch.utils.data.Dataset):
    """Lazy 24-hour window dataset with explicit missing mask + time metadata."""

    def __init__(
        self,
        split_df,
        feature_cols,
        window_size=24,
        station_structural_missing_map=None,
        all_data_selection=True,
    ):
        self.feature_cols = feature_cols
        self.window_size = window_size
        self.station_data = {}
        self.window_index = []
        self.dropped_gap_windows = 0
        self.dropped_complete_windows = 0
        self.total_contiguous_endpoints = 0
        self.station_structural_missing_map = station_structural_missing_map or {}

        self.all_data_selection = bool(all_data_selection)

        for station_id, station_df in split_df.groupby("StationId", sort=False):
            station_df = station_df.sort_values("Datetime").reset_index(drop=True)
            if len(station_df) < window_size:
                continue

            features_np = station_df[feature_cols].to_numpy(dtype=np.float32)

            # Endpoint is valid only if the previous (window_size - 1) gaps are exactly 1 hour.
            diffs = station_df["Datetime"].diff().dt.total_seconds()
            contiguous = diffs.eq(3600)
            valid_end = (
                contiguous.rolling(window=window_size - 1, min_periods=window_size - 1)
                .sum()
                .eq(window_size - 1)
            )
            valid_end = valid_end.fillna(False)
            contiguous_end_indices = np.flatnonzero(valid_end.to_numpy())
            self.total_contiguous_endpoints += len(contiguous_end_indices)
            if len(contiguous_end_indices) == 0:
                continue

            selected_end_indices = contiguous_end_indices
            if not self.all_data_selection:
                structural_missing = self.station_structural_missing_map.get(station_id)
                if structural_missing is None:
                    structural_missing = np.zeros(len(feature_cols), dtype=bool)

                effective_missing = np.isnan(features_np) & (
                    ~structural_missing[None, :]
                )
                row_is_incomplete = effective_missing.any(axis=1)
                selected_end_indices = contiguous_end_indices[
                    row_is_incomplete[contiguous_end_indices]
                ]

            if len(selected_end_indices) == 0:
                continue

            self.station_data[station_id] = {
                "features": features_np,
                "hour": station_df["Datetime"].dt.hour.to_numpy(dtype=np.int16),
                "month": station_df["Datetime"].dt.month.to_numpy(dtype=np.int16),
                "label": station_df["label"].to_numpy(dtype=np.int64),
                "datetime": station_df["Datetime"].to_numpy(),
            }

            for end_idx in selected_end_indices:
                self.window_index.append((station_id, int(end_idx)))

            possible_endpoints = max(len(station_df) - window_size + 1, 0)
            self.dropped_gap_windows += possible_endpoints - len(contiguous_end_indices)
            self.dropped_complete_windows += len(contiguous_end_indices) - len(
                selected_end_indices
            )

    def __len__(self):
        return len(self.window_index)

    def __getitem__(self, idx):
        station_id, end_idx = self.window_index[idx]
        station = self.station_data[station_id]
        start_idx = end_idx - self.window_size + 1

        raw = station["features"][start_idx : end_idx + 1]
        missing_mask = np.isnan(raw).astype(np.uint8)
        hours = station["hour"][start_idx : end_idx + 1]
        months = station["month"][start_idx : end_idx + 1]
        label = int(station["label"][end_idx])

        return {
            "raw": raw,  # [24, 7]
            "missing_mask": missing_mask,  # [24, 7]
            "hour": hours,  # [24]
            "month": months,  # [24]
            "label": label,
            "station_id": station_id,
            "end_datetime": station["datetime"][end_idx],
        }


foundation_train = AQIWindowDataset(
    split_train_df,
    FEATURES,
    window_size=WINDOW_SIZE,
    station_structural_missing_map=station_structural_missing_map,
    all_data_selection=ALL_DATA_SELECTION,
)
foundation_val = AQIWindowDataset(
    split_val_df,
    FEATURES,
    window_size=WINDOW_SIZE,
    station_structural_missing_map=station_structural_missing_map,
    all_data_selection=ALL_DATA_SELECTION,
)
foundation_test = AQIWindowDataset(
    split_test_df,
    FEATURES,
    window_size=WINDOW_SIZE,
    station_structural_missing_map=station_structural_missing_map,
    all_data_selection=ALL_DATA_SELECTION,
)

print(
    f"Time cutoffs -> train: <= {train_cutoff}, val: <= {val_cutoff}, test: > {val_cutoff}"
)
print(f"ALL_DATA_SELECTION: {ALL_DATA_SELECTION}")
print(
    f"Window samples -> train: {len(foundation_train):,}, val: {len(foundation_val):,}, test: {len(foundation_test):,}"
)
print(
    "Contiguous endpoints before completeness filter -> "
    f"train: {foundation_train.total_contiguous_endpoints:,}, "
    f"val: {foundation_val.total_contiguous_endpoints:,}, "
    f"test: {foundation_test.total_contiguous_endpoints:,}"
)
print(
    "Contiguous endpoints total (all splits): "
    f"{(foundation_train.total_contiguous_endpoints + foundation_val.total_contiguous_endpoints + foundation_test.total_contiguous_endpoints):,}"
)
print(
    "Dropped windows due to gaps -> "
    f"train: {foundation_train.dropped_gap_windows:,}, "
    f"val: {foundation_val.dropped_gap_windows:,}, "
    f"test: {foundation_test.dropped_gap_windows:,}"
)
if not ALL_DATA_SELECTION:
    print(
        "Dropped windows by completeness filter -> "
        f"train: {foundation_train.dropped_complete_windows:,}, "
        f"val: {foundation_val.dropped_complete_windows:,}, "
        f"test: {foundation_test.dropped_complete_windows:,}"
    )
print("Stations with structural-missing channels per pollutant:")
print(station_structural_missing.sum().to_dict())

if len(foundation_train) > 0:
    sample = foundation_train[0]
    print("Sample contract shapes:")
    print(
        f"raw={sample['raw'].shape}, missing_mask={sample['missing_mask'].shape}, hour={sample['hour'].shape}, month={sample['month'].shape}, label={sample['label']}"
    )


Time cutoffs -> train: <= 2019-05-26T12:00:00.000000, val: <= 2019-12-13T06:00:00.000000, test: > 2019-12-13T06:00:00.000000
ALL_DATA_SELECTION: False
Window samples -> train: 279,205, val: 58,352, test: 89,591
Contiguous endpoints before completeness filter -> train: 1,090,299, val: 395,102, test: 441,346
Contiguous endpoints total (all splits): 1,926,747
Dropped windows due to gaps -> train: 56,115, val: 15,048, test: 14,466
Dropped windows by completeness filter -> train: 811,094, val: 336,750, test: 351,755
Stations with structural-missing channels per pollutant:
{'PM2.5': 2, 'PM10': 17, 'SO2': 9, 'NOx': 2, 'NH3': 26, 'CO': 0, 'O3': 7}
Sample contract shapes:
raw=(24, 7), missing_mask=(24, 7), hour=(24,), month=(24,), label=4


## Breakpoint-aware Gaussian population (absolute channels)

Deterministic spike latency encoding based on Gaussian bell-curve sensitivity inside each range. It returns only absolute-channel spikes for now.

In [17]:
POPULATION_STEPS = POPULATION_STEPS_ACTIVE
POPULATION_EPSILON = 0.05
POPULATION_WIDTH_SCALE = 0.35

# AQI-aligned breakpoint edges per pollutant.
# Last edge is a practical severe-tail cap; it is expanded with train q99 if needed.
BASE_BREAKPOINTS = {
    "PM2.5": [0.0, 30.0, 60.0, 90.0, 120.0, 250.0, 380.0],
    "PM10": [0.0, 50.0, 100.0, 250.0, 350.0, 430.0, 510.0],
    "SO2": [0.0, 40.0, 80.0, 380.0, 800.0, 1600.0, 2400.0],
    "NOx": [0.0, 40.0, 80.0, 180.0, 280.0, 400.0, 520.0],
    "NH3": [0.0, 200.0, 400.0, 800.0, 1200.0, 1800.0, 2400.0],
    "CO": [0.0, 1.0, 2.0, 10.0, 17.0, 34.0, 51.0],
    "O3": [0.0, 50.0, 100.0, 168.0, 208.0, 748.0, 1287.0],
}


def build_population_breakpoint_config(
    train_df,
    feature_cols,
    base_breakpoints,
    width_scale=0.35,
):
    """Build per-pollutant breakpoint config: midpoint centers + local-width Gaussian sigmas."""
    q99 = train_df[feature_cols].quantile(0.99)
    config = {}

    for feat in feature_cols:
        edges = np.asarray(base_breakpoints[feat], dtype=np.float32).copy()
        upper_cap = q99.get(feat, np.nan)
        if pd.isna(upper_cap):
            upper_cap = edges[-1]
        edges[-1] = max(float(edges[-1]), float(upper_cap))

        interval_widths = np.diff(edges)
        centers = edges[:-1] + 0.5 * interval_widths
        widths = np.maximum(width_scale * interval_widths, 1e-6)

        config[feat] = {
            "edges": edges,
            "centers": centers.astype(np.float32),
            "widths": widths.astype(np.float32),
            "clip_min": float(edges[0]),
            "clip_max": float(edges[-1]),
        }

    return config


# [24*S, C_abs]
def encode_absolute_population_latency(
    sample,
    config,
    feature_cols,
    micro_steps=8,
    epsilon=0.05,
):
    raw = sample["raw"].astype(np.float32)
    missing_mask = sample["missing_mask"].astype(bool)
    hours = raw.shape[0]
    total_channels = sum(len(config[feat]["centers"]) for feat in feature_cols)

    spikes = np.zeros((hours * micro_steps, total_channels), dtype=np.uint8)
    channel_slices = {}
    channel_offset = 0

    for feat_idx, feat in enumerate(feature_cols):
        feat_cfg = config[feat]
        centers = feat_cfg["centers"][None, :]  # [1, Jp]
        widths = feat_cfg["widths"][None, :]  # [1, Jp]

        # prevent extreme outliers from breaking the Gaussian calculation
        x = np.clip(raw[:, feat_idx], feat_cfg["clip_min"], feat_cfg["clip_max"])
        x = x[:, None]  # [24, 1]
        # calculate Gaussian: the closer to center, the earlier the spike (lower latency), or vice versa
        activations = np.exp(-0.5 * ((x - centers) / widths) ** 2).astype(np.float32)
        activations[missing_mask[:, feat_idx], :] = 0.0

        # if activation is too low (far from all centers), no spike; otherwise calculate latency based on activation strength
        active = activations >= epsilon
        tau = 1 + np.floor((1.0 - activations) * (micro_steps - 1)).astype(np.int32)

        j_count = centers.shape[1]

        # calulate where to put spike on global matrix based on hour, latency, and channel offset for the feature
        for h in range(hours):
            if not np.any(active[h]):
                continue
            active_j = np.flatnonzero(active[h])
            global_t = (h * micro_steps) + (tau[h, active_j] - 1)
            spikes[global_t, channel_offset + active_j] = 1

        # remember which columns belong to this pollutant for later indexing in the model
        channel_slices[feat] = slice(channel_offset, channel_offset + j_count)
        channel_offset += j_count

    return spikes, channel_slices


class AbsolutePopulationSpikeDataset(torch.utils.data.Dataset):
    def __init__(
        self, foundation_dataset, config, feature_cols, micro_steps=8, epsilon=0.05
    ):
        self.foundation_dataset = foundation_dataset
        self.config = config
        self.feature_cols = feature_cols
        self.micro_steps = micro_steps
        self.epsilon = epsilon
        self.num_channels = sum(len(config[f]["centers"]) for f in feature_cols)

    def __len__(self):
        return len(self.foundation_dataset)

    def __getitem__(self, idx):
        sample = self.foundation_dataset[idx]
        spikes_abs, channel_slices = encode_absolute_population_latency(
            sample,
            self.config,
            self.feature_cols,
            micro_steps=self.micro_steps,
            epsilon=self.epsilon,
        )

        return {
            "spikes_abs": spikes_abs,  # [24*S, C_abs]
            "label": sample["label"],
            "station_id": sample["station_id"],
            "end_datetime": sample["end_datetime"],
            "channel_slices": channel_slices,
        }


population_config = build_population_breakpoint_config(
    split_train_df,
    FEATURES,
    BASE_BREAKPOINTS,
    width_scale=POPULATION_WIDTH_SCALE,
)

absolute_train = AbsolutePopulationSpikeDataset(
    foundation_train,
    population_config,
    FEATURES,
    micro_steps=POPULATION_STEPS,
    epsilon=POPULATION_EPSILON,
)
absolute_val = AbsolutePopulationSpikeDataset(
    foundation_val,
    population_config,
    FEATURES,
    micro_steps=POPULATION_STEPS,
    epsilon=POPULATION_EPSILON,
)
absolute_test = AbsolutePopulationSpikeDataset(
    foundation_test,
    population_config,
    FEATURES,
    micro_steps=POPULATION_STEPS,
    epsilon=POPULATION_EPSILON,
)

print("Population breakpoint config summary:")
for feat in FEATURES:
    cfg = population_config[feat]
    print(
        f"- {feat}: neurons={len(cfg['centers'])}, clip=[{cfg['clip_min']:.2f}, {cfg['clip_max']:.2f}], first_center={cfg['centers'][0]:.2f}"
    )

print(
    f"Absolute-only datasets -> train: {len(absolute_train):,}, val: {len(absolute_val):,}, test: {len(absolute_test):,}"
)
if len(absolute_train) > 0:
    sample_abs = absolute_train[0]
    spikes_abs = sample_abs["spikes_abs"]
    spikes_abs_repeat, _ = encode_absolute_population_latency(
        foundation_train[0],
        population_config,
        FEATURES,
        micro_steps=POPULATION_STEPS,
        epsilon=POPULATION_EPSILON,
    )
    print(
        f"Absolute spike tensor shape: {spikes_abs.shape} (T={WINDOW_SIZE * POPULATION_STEPS}, C_abs={absolute_train.num_channels})"
    )
    print(f"Total spikes in sample: {int(spikes_abs.sum())}")
    print(
        f"Deterministic check (same sample, same config): {np.array_equal(spikes_abs, spikes_abs_repeat)}"
    )

Population breakpoint config summary:
- PM2.5: neurons=6, clip=[0.00, 464.00], first_center=15.00
- PM10: neurons=6, clip=[0.00, 755.75], first_center=25.00
- SO2: neurons=6, clip=[0.00, 2400.00], first_center=20.00
- NOx: neurons=6, clip=[0.00, 520.00], first_center=20.00
- NH3: neurons=6, clip=[0.00, 2400.00], first_center=100.00
- CO: neurons=6, clip=[0.00, 51.00], first_center=0.50
- O3: neurons=6, clip=[0.00, 1287.00], first_center=25.00
Absolute-only datasets -> train: 279,205, val: 58,352, test: 89,591
Absolute spike tensor shape: (96, 42) (T=96, C_abs=42)
Total spikes in sample: 277
Deterministic check (same sample, same config): True


In [18]:
# SNN model class used by the population pipeline
class AQI_SNN(nn.Module):
    def __init__(self, num_features=7, num_classes=6):
        super().__init__()
        self.fc1 = nn.Linear(num_features, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc_out = nn.Linear(64, num_classes)

        # SNN Change 2
        self.lif1 = snn.Leaky(beta=0.95)
        self.lif2 = snn.Leaky(beta=0.90)
        self.lif_out = snn.Leaky(beta=0.85)

    # spikes: [NUM_STEPS, batch, 7]
    def forward(self, spikes):
        T = spikes.shape[0]
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem_out = self.lif_out.init_leaky()
        out_rec = torch.zeros(
            spikes.shape[1], self.fc_out.out_features, device=spikes.device
        )

        # SNN Change 3
        for t in range(T):
            x = spikes[t]
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            cur_out = self.fc_out(spk2)
            spk_out, mem_out = self.lif_out(cur_out, mem_out)
            out_rec += spk_out

        return out_rec

In [19]:
# Deterministic Gaussian absolute encoding
def collate_population_batch(batch):
    spikes = torch.tensor(
        np.stack([item["spikes_abs"] for item in batch]), dtype=torch.float32
    )
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return spikes, labels


def extract_window_labels(dataset):
    labels = np.empty(len(dataset), dtype=np.int64)
    for i, (station_id, end_idx) in enumerate(dataset.window_index):
        labels[i] = dataset.station_data[station_id]["label"][end_idx]
    return labels


if len(absolute_train) == 0:
    raise ValueError("absolute_train is empty; cannot train population-only path.")

pin_memory = torch.cuda.is_available()
persistent_workers = NUM_WORKERS > 0

pop_train_loader = DataLoader(
    absolute_train,
    batch_size=TRAIN_BATCH_SIZE_ACTIVE,
    shuffle=True,
    collate_fn=collate_population_batch,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)
pop_val_loader = DataLoader(
    absolute_val,
    batch_size=EVAL_BATCH_SIZE_ACTIVE,
    shuffle=False,
    collate_fn=collate_population_batch,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)
pop_test_loader = DataLoader(
    absolute_test,
    batch_size=EVAL_BATCH_SIZE_ACTIVE,
    shuffle=False,
    collate_fn=collate_population_batch,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)

device_pop = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device (population path): {device_pop}")

# Reuse same SNN class, but with expanded input channels
model_pop = AQI_SNN(num_features=absolute_train.num_channels, num_classes=6).to(
    device_pop
)

pop_train_labels = extract_window_labels(foundation_train)
pop_class_counts = np.bincount(pop_train_labels, minlength=6).astype(np.float64)
pop_class_weights = 1.0 / np.sqrt(np.maximum(pop_class_counts, 1.0))
pop_class_weights = pop_class_weights / pop_class_weights.sum() * len(pop_class_weights)
criterion_pop = nn.CrossEntropyLoss(
    weight=torch.tensor(pop_class_weights, dtype=torch.float32).to(device_pop)
)
print(
    f"Population path class weights: {dict(zip([INT_TO_BUCKET[i] for i in range(6)], pop_class_weights.round(2)))}"
)

optimizer_pop = torch.optim.Adam(model_pop.parameters(), lr=1e-3)
scheduler_pop = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_pop, patience=2, factor=0.5
)

if DEV_MODE:
    print(
        f"[DEV] Fast profile active: epochs={EPOCHS_POP_ACTIVE}, train_batch_cap={MAX_TRAIN_BATCHES_ACTIVE}, eval_batch_cap={MAX_EVAL_BATCHES_ACTIVE}"
    )

EPOCHS_POP = EPOCHS_POP_ACTIVE
for epoch in range(EPOCHS_POP):
    model_pop.train()
    total_loss, correct, total = 0.0, 0, 0
    processed_batches = 0

    for batch_idx, (spikes_batch, y_batch) in enumerate(pop_train_loader):
        if MAX_TRAIN_BATCHES_ACTIVE is not None and batch_idx >= MAX_TRAIN_BATCHES_ACTIVE:
            break

        spikes_batch = spikes_batch.to(device_pop)
        y_batch = y_batch.to(device_pop)

        # spikes_batch: [batch, T, C_abs] -> model expects [T, batch, C_abs]
        spike_data = spikes_batch.permute(1, 0, 2).contiguous()

        out = model_pop(spike_data)
        loss = criterion_pop(out, y_batch)

        optimizer_pop.zero_grad()
        loss.backward()
        optimizer_pop.step()

        total_loss += loss.item() * len(y_batch)
        correct += (out.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
        processed_batches += 1

    if processed_batches == 0:
        raise ValueError("No training batches were processed. Increase batch cap or dataset size.")

    avg_loss = total_loss / max(total, 1)
    scheduler_pop.step(avg_loss)
    print(
        f"[Population] Epoch {epoch+1:2d}/{EPOCHS_POP}  batches={processed_batches}  loss={avg_loss:.4f}  acc={correct/max(total,1):.3f}  lr={optimizer_pop.param_groups[0]['lr']:.1e}"
    )

Device (population path): cpu
Population path class weights: {'Good': np.float64(1.45), 'Satisfactory': np.float64(0.77), 'Moderate': np.float64(0.71), 'Poor': np.float64(1.04), 'Very Poor': np.float64(0.89), 'Severe': np.float64(1.13)}
[DEV] Fast profile active: epochs=3, train_batch_cap=80, eval_batch_cap=30
[Population] Epoch  1/3  batches=80  loss=1.4306  acc=0.398  lr=1.0e-03
[Population] Epoch  2/3  batches=80  loss=0.7635  acc=0.696  lr=1.0e-03
[Population] Epoch  3/3  batches=80  loss=0.6904  acc=0.727  lr=1.0e-03


In [20]:
### Evaluation on population-path test set
model_pop.eval()
all_preds_pop, all_labels_pop = [], []

with torch.no_grad():
    for batch_idx, (spikes_batch, y_batch) in enumerate(pop_test_loader):
        if MAX_EVAL_BATCHES_ACTIVE is not None and batch_idx >= MAX_EVAL_BATCHES_ACTIVE:
            break

        spikes_batch = spikes_batch.to(device_pop)
        spike_data = spikes_batch.permute(1, 0, 2).contiguous()
        out = model_pop(spike_data)

        all_preds_pop.extend(out.argmax(1).cpu().tolist())
        all_labels_pop.extend(y_batch.tolist())

present_pop = sorted(set(all_labels_pop) | set(all_preds_pop))
target_names_pop = [INT_TO_BUCKET[i] for i in present_pop]

acc_pop = sum(p == l for p, l in zip(all_preds_pop, all_labels_pop)) / max(
    len(all_labels_pop), 1
)
print(f"\nPopulation-path test accuracy: {acc_pop:.3f}")
print(
    f"Population-path baseline (most-frequent class): {max(all_labels_pop.count(c) for c in present_pop) / max(len(all_labels_pop), 1):.3f}\n"
)
print(
    classification_report(
        all_labels_pop,
        all_preds_pop,
        labels=present_pop,
        target_names=target_names_pop,
    )
)
print("Population-path confusion matrix:")
print(confusion_matrix(all_labels_pop, all_preds_pop, labels=present_pop))


Population-path test accuracy: 0.754
Population-path baseline (most-frequent class): 0.367

              precision    recall  f1-score   support

        Good       0.72      0.91      0.80      1282
Satisfactory       0.67      0.84      0.75      3560
    Moderate       0.84      0.72      0.78      5631
        Poor       0.62      0.55      0.58      1729
   Very Poor       0.90      0.69      0.78      2239
      Severe       0.72      0.92      0.81       919

    accuracy                           0.75     15360
   macro avg       0.75      0.77      0.75     15360
weighted avg       0.77      0.75      0.75     15360

Population-path confusion matrix:
[[1171  111    0    0    0    0]
 [ 417 3001  136    1    5    0]
 [  49 1278 4062  216   13   13]
 [   0   68  626  952   77    6]
 [   0    0   15  364 1556  304]
 [   0    0    0    0   76  843]]
